In [3]:
import re
import pandas as pd


def parse_can_log(path):
    """
    Parse CAN log lines like:
    (1290000000.012715) can0 230#FD000002D4000400
    Returns a DataFrame with columns: Timestamp (float), ID (hex str), Data (hex str), ID_int (int)
    """
    pattern = re.compile(r'^\((?P<ts>\d+\.\d+)\)\s+\S+\s+(?P<id>[0-9A-Fa-f]+)#(?P<data>[0-9A-Fa-f]+)')
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            m = pattern.match(line.strip())
            if not m:
                continue
            ts = float(m.group('ts'))
            id_hex = m.group('id').upper()
            data_hex = m.group('data').upper()
            try:
                id_int = int(id_hex, 16)
            except ValueError:
                id_int = None
            rows.append((ts, id_hex, data_hex, id_int))
    return pd.DataFrame(rows, columns=['Timestamp', 'ID', 'Data', 'ID_int'])

# Example usage:
df_attack = parse_can_log('C:\\Users\\nb0801\\Documents\\GitHub\\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\\datasets\\road_dataset\\road\\attacks\\fuzzing_attack_2.log')
df_attack["Attack"] = 'R'
print(df_attack.head())

      Timestamp   ID              Data  ID_int Attack
0  1.010000e+09  00E  20529602080975B8      14      R
1  1.010000e+09  153  00000004000C0004     339      R
2  1.010000e+09  295  0000000000000040     661      R
3  1.010000e+09  662  4EE0000040000000    1634      R
4  1.010000e+09  19C  08F62000020028C0     412      R


In [4]:
# set Attack to 'T' for timestamps in the given interval
df_attack.loc[
    (df_attack['Timestamp'] >= 1010000000.000000 + 11.367798) & (df_attack['Timestamp'] <= 1010000000.000000 + 13.346811),
    'Attack'
] = 'T'

In [4]:
import numpy as np

def sliding_windows_id_data(df, window_size=32, step=2, attack_label='T'):
    ids = df['ID'].to_numpy(dtype=object)
    data = df['Data'].to_numpy(dtype=object)
    attack = df['Attack'].to_numpy(dtype=object)

    n = len(df)
    num_windows = (n - window_size) // step + 1
    windows = []
    window_labels = []

    for start in range(0, n - window_size + 1, step):
        end = start + window_size
        windows.append(np.column_stack((ids[start:end], data[start:end])))
        window_labels.append(attack_label if 'T' in attack[start:end] else 'R')

    return windows, np.array(window_labels, dtype=object)


In [5]:
windowed_id_data, window_attack_labels = sliding_windows_id_data(df_attack, window_size=32, step=2, attack_label='F')
windowed_id_data

[array([['00E', '20529602080975B8'],
        ['153', '00000004000C0004'],
        ['295', '0000000000000040'],
        ['662', '4EE0000040000000'],
        ['19C', '08F62000020028C0'],
        ['0D0', '22770460F8000000'],
        ['033', '0007C0000D0007D0'],
        ['274', '00F224FB507C81DD'],
        ['0C0', '0000000000000000'],
        ['30A', '402D5001D32A3200'],
        ['193', '00080803E8880000'],
        ['20E', '4E2003A0003F6FFF'],
        ['522', 'DF7FD1AC4DC8DF9C'],
        ['366', '7FDDC8020147CE78'],
        ['354', '1FFF40000003FA00'],
        ['5E1', '893FE00B0A000080'],
        ['28B', '0000000000000000'],
        ['580', '00000003B4880000'],
        ['434', '0382A0A000000000'],
        ['6E0', '0000000000000000'],
        ['0A7', '2010FA24D12B80A0'],
        ['2B4', '14E9950000005800'],
        ['1CA', '3FF1FF8000000C38'],
        ['1C4', '0000020001000000'],
        ['2A4', '36DC001C040898D0'],
        ['498', '87FFBFFCEE80000E'],
        ['32D', '80000420AC010020'],
 

In [6]:
def format_id_data_windows(windows):
    formatted_all = []
    for window in windows:
        window = np.asarray(window, dtype=object)
        formatted = []
        for msg_id, data in window:
            id_str = str(msg_id).strip().lower().zfill(4)
            data_str = str(data).strip().lower()
            data_str = re.sub(r'[^0-9a-f]', '', data_str)
            if len(data_str) % 2:
                data_str = '0' + data_str
            data_bytes = [data_str[i:i+2] for i in range(0, len(data_str), 2)] if data_str else []
            formatted.append([id_str, ' '.join(data_bytes)])
        formatted_all.append(np.array(formatted, dtype=object))
    return np.array(formatted_all, dtype=object)

formatted_windows = format_id_data_windows(windowed_id_data)
formatted_windows

array([[['000e', '20 52 96 02 08 09 75 b8'],
        ['0153', '00 00 00 04 00 0c 00 04'],
        ['0295', '00 00 00 00 00 00 00 40'],
        ...,
        ['069e', '04 40 04 7f 1f c0 13 e2'],
        ['0fff', '00 00 00 00 00 00 00 00'],
        ['0125', '90 00 40 9f 42 be 31 60']],

       [['0295', '00 00 00 00 00 00 00 40'],
        ['0662', '4e e0 00 00 40 00 00 00'],
        ['019c', '08 f6 20 00 02 00 28 c0'],
        ...,
        ['0125', '90 00 40 9f 42 be 31 60'],
        ['000e', '20 52 96 02 08 09 75 b2'],
        ['00d0', '2a 77 04 60 f7 00 00 00']],

       [['019c', '08 f6 20 00 02 00 28 c0'],
        ['00d0', '22 77 04 60 f8 00 00 00'],
        ['0033', '00 07 c0 00 0d 00 07 d0'],
        ...,
        ['00d0', '2a 77 04 60 f7 00 00 00'],
        ['0033', '00 07 b8 00 0d 40 07 d0'],
        ['0577', '00 00 08 00 00 00 01 1a']],

       ...,

       [['0fff', '00 00 00 00 00 00 00 00'],
        ['00ba', '06 a8 73 84 10 00 06 2c'],
        ['0297', '11 01 00 20 01 02 51 10'

In [7]:
def _hex_byte_list_from_row(row):
    msg_id, data = row
    msg_id = str(msg_id).strip()
    if len(msg_id) % 2 != 0:
        msg_id = '0' + msg_id
    id_bytes = [msg_id[i:i+2] for i in range(0, len(msg_id), 2)]
    data_bytes = [b for b in str(data).split() if b]
    return [int(b, 16) for b in id_bytes + data_bytes]

def convert_windowed_id_data_to_ints(windowed_data, pad_value=0):
    int_windows = []
    for window in windowed_data:
        rows = [_hex_byte_list_from_row(row) for row in window]
        max_len = max(len(r) for r in rows)
        padded_rows = [r + [pad_value] * (max_len - len(r)) for r in rows]
        int_windows.append(np.array(padded_rows, dtype=np.uint8))
    return int_windows

int_windowed_id_data = convert_windowed_id_data_to_ints(formatted_windows)

int_windowed_id_data = np.array(int_windowed_id_data)


In [8]:
window_attack_onehot = np.zeros((len(window_attack_labels), 4), dtype=np.uint8)
window_attack_onehot[window_attack_labels == 'R', 0] = 1
window_attack_onehot[window_attack_labels == 'D', 1] = 1
window_attack_onehot[window_attack_labels == 'F', 2] = 1
window_attack_onehot[window_attack_labels == 'S', 3] = 1

print(window_attack_onehot)


[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 ...
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [9]:
import json
from tensorflow.keras.models import model_from_json

json_path = "model_architecture.json"
weights_path = "model_weights.weights.h5"

with open(json_path, 'r', encoding='utf-8') as f:
    model_json = f.read()

model = model_from_json(model_json)
model.load_weights(weights_path)

# compile if you plan to evaluate/train; not required for inference
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Model loaded.")
model.summary()

Model loaded.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 30, 8, 32)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 15, 4, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 2, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1664)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       106,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,636 (490.77 KB)

 Trainable params: 125,636 (490.77 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
loss, accuracy = model.evaluate(int_windowed_id_data, window_attack_onehot, verbose=0)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

label_map = np.array(['R', 'D', 'F', 'S'])

def onehot_to_labels(onehot):
    return label_map[np.argmax(onehot, axis=1)]

y_pred_probs = model.predict(int_windowed_id_data)
y_pred = onehot_to_labels(y_pred_probs)
y_true = onehot_to_labels(window_attack_onehot)

report = classification_report(
    y_true,
    y_pred,
    labels=label_map,
    target_names=label_map,
    zero_division=0,
)

cm = confusion_matrix(y_true, y_pred, labels=label_map)

tps = np.diag(cm)
fps = cm.sum(axis=0) - tps
fns = cm.sum(axis=1) - tps
tns = cm.sum() - (tps + fps + fns)

fpr = np.divide(fps, fps + tns, out=np.zeros_like(fps, dtype=float), where=(fps + tns) != 0)
fnr = np.divide(fns, fns + tps, out=np.zeros_like(fns, dtype=float), where=(fns + tps) != 0)

print("Classification report:")
print(report)
print("Confusion matrix:")
print(cm)

precision = precision_score(y_true, y_pred, labels=label_map, average=None, zero_division=0)
recall = recall_score(y_true, y_pred, labels=label_map, average=None, zero_division=0)
f1 = f1_score(y_true, y_pred, labels=label_map, average=None, zero_division=0)

for label, p, r, f, fp_rate, fn_rate in zip(label_map, precision, recall, f1, fpr, fnr):
    print(
        f"{label}: precision={p:.4f}, recall={r:.4f}, "
        f"f1={f:.4f}, false positive rate={fp_rate:.4f}, false negative rate={fn_rate:.4f}"
    )

Test loss: 14.2109
Test accuracy: 0.1694
505/505 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Classification report:
              precision    recall  f1-score   support

           R       1.00      0.01      0.03     13595
           D       0.00      0.00      0.00         0
           F       0.16      1.00      0.28      2565
           S       0.00      0.00      0.00         0

    accuracy                           0.17     16160
   macro avg       0.29      0.25      0.08     16160
weighted avg       0.87      0.17      0.07     16160

Confusion matrix:
[[  173     1 13287   134]
 [    0     0     0     0]
 [    0     0  2565     0]
 [    0     0     0     0]]
R: precision=1.0000, recall=0.0127, f1=0.0251, false positive rate=0.0000, false negative rate=0.9873
D: precision=0.0000, recall=0.0000, f1=0.0000, false positive rate=0.0001, false negative rate=0.0000
F: precision=0.1618, recall=1.0000, f1=0.2785, false positive rate=0.9773, false negative rate=0.0000
S: precision=0.0000, recall